编码器-解码器（Encoder-Decoder）架构是一种用于处理输入和输出均为可变长度序列的通用模型结构，广泛应用于机器翻译等序列转换任务。其核心思想是：先由编码器将输入序列（如源语言句子）压缩为一个固定形状的“上下文表示”（状态），再由解码器基于该表示逐步生成目标序列（如翻译后的句子）。其中，编码器负责提取输入序列的信息，解码器则根据该信息按时间步逐个生成输出，是后续各类序列模型（如注意力机制、Transformer）的基础框架。

在编码器接口中，我们只指定长度可变的序列作为编码器的输入X。 任何继承这个Encoder基类的模型将完成代码实现。

In [1]:
from torch import nn


class Encoder(nn.Module):
    """编码器-解码器架构的基本编码器接口"""
    def __init__(self, **kwargs):
        super(Encoder, self).__init__(**kwargs)

    def forward(self, X, *args):
        raise NotImplementedError

解码器接口中，我们新增一个init_state函数， 用于将编码器的输出（enc_outputs）转换为编码后的状态。

为了逐个地生成长度可变的词元序列， 解码器在每个时间步都会将输入 （例如：在前一时间步生成的词元）和编码后的状态 映射成当前时间步的输出词元。

In [2]:
class Decoder(nn.Module):
    """编码器-解码器架构的基本解码器接口"""
    def __init__(self, **kwargs):
        super(Decoder, self).__init__(**kwargs)

    def init_state(self, enc_outputs, *args):
        raise NotImplementedError

    def forward(self, X, state):
        raise NotImplementedError

合并编码器和解码器

编码器-解码器”架构包含了一个编码器和一个解码器， 并且还拥有可选的额外的参数。 在前向传播中，编码器的输出用于生成编码状态， 这个状态又被解码器作为其输入的一部分。

In [3]:
class EncoderDecoder(nn.Module):
    """编码器-解码器架构的基类"""
    def __init__(self, encoder, decoder, **kwargs):
        super(EncoderDecoder, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, enc_X, dec_X, *args):
        enc_outputs = self.encoder(enc_X, *args)
        dec_state = self.decoder.init_state(enc_outputs, *args)
        return self.decoder(dec_X, dec_state)